In [1]:
# pip install langchain langchain_openai langchain_community chromadb

In [2]:
# from google.colab import drive
# import os

# # 먼저 구글 드라이브 마운트
# drive.mount('/content/drive')

In [3]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")

# 환경 변수에서 API 키 가져오기
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [4]:

# 쿼리를 위한 로그 설정
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)


In [5]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 로더 설정
loaders = [TextLoader("D:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\Data\How_to_invest_money.txt", encoding="utf-8")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

d:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 문서 생성을 위한 텍스트 분할기 정의
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 문서 분할
split_docs = recursive_splitter.split_documents(docs)

# OpenAIEmbeddings 인스턴스 생성
embeddings = OpenAIEmbeddings()

# Chroma vectorstore 생성
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

In [7]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# LLM 모델 설정 (여기서는 ChatOpenAI 사용)
llm = ChatOpenAI(model = "gpt-4o",temperature=0.2)

# MultiQueryRetriever 실행
retriever = MultiQueryRetriever.from_llm(
retriever=vectorstore.as_retriever(), # 기본 검색기 (벡터 데이터베이스)
llm=llm, # 앞서 정의한 llm (gpt-4o)
)

In [8]:
# 샘플 질문
question = "주식 투자를 처음 시작하려면 어떻게 해야 하나요?"

# 결과 검색
unique_docs = retriever.invoke(question)
print(f"\n결과: {len(unique_docs)}개의 문서가 검색되었습니다.")


결과: 5개의 문서가 검색되었습니다.


In [9]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

# RetrievalQA 체인 설정
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# 질문에 대한 답변 생성
result = qa_chain.invoke({"query": question})

# 결과 출력
print("답변:", result["result"])
print("\n사용된 문서:")
for doc in result["source_documents"]:
    print(doc.page_content)

답변: 주식 투자를 처음 시작하려면 다음과 같은 단계들을 고려해볼 수 있습니다:

1. **기초 지식 습득**: 주식 시장의 기본 원리와 주식의 작동 방식을 이해하는 것이 중요합니다. 주식과 채권의 차이점, 주식 시장의 작동 방식 등을 학습하세요.

2. **목표 설정**: 투자 목표를 명확히 설정하세요. 장기적인 자산 증식을 목표로 할 것인지, 단기적인 수익을 추구할 것인지 결정해야 합니다.

3. **재정 상태 평가**: 투자 가능한 자금을 평가하고, 비상금과 같은 필수 자금을 제외한 여유 자금을 투자에 사용할 수 있도록 준비하세요.

4. **증권 계좌 개설**: 주식 거래를 위해 증권 계좌를 개설해야 합니다. 은행이나 증권사를 통해 계좌를 개설할 수 있습니다.

5. **시장 조사**: 관심 있는 산업이나 기업에 대해 조사하고, 그들의 재무 상태와 시장 위치를 분석하세요.

6. **포트폴리오 구성**: 위험 분산을 위해 다양한 주식에 투자하여 포트폴리오를 구성하세요. 이는 특정 주식의 가격 변동에 따른 리스크를 줄이는 데 도움이 됩니다.

7. **투자 전략 수립**: 장기 투자, 단기 매매, 배당 수익 추구 등 자신의 투자 스타일에 맞는 전략을 수립하세요.

8. **지속적인 학습**: 시장 동향과 경제 뉴스에 지속적으로 관심을 가지고 학습하세요. 이는 투자 결정을 내리는 데 도움이 됩니다.

9. **감정 관리**: 주식 시장은 변동성이 크기 때문에 감정에 휘둘리지 않고 냉정하게 투자 결정을 내리는 것이 중요합니다.

이러한 단계들을 통해 주식 투자를 시작할 수 있으며, 경험을 쌓으면서 점차 자신의 투자 방식을 발전시킬 수 있습니다.

사용된 문서:
After learning how to judge the value of every form of investment, a man
may still be unsuccessful in the investment of money unless he acquires
also a firm grasp upon t